# 02 — Feature Engineering
**RoadSentinel AI** — Turn raw vehicle tracking coordinates into kinematic and overlap features.

In [ ]:
import os, sys
sys.path.insert(0, '..')
import pandas as pd, numpy as np
from src.features import engineer_motion_features, iou_overlap_features, summarize_video_features
from src.detection import extract_tracks

## 1. Extract Tracks or Load Sample Trajectory Data

In [ ]:
sample_clip = '../demo/clip_1.mp4'
if not os.path.exists(sample_clip):
    from scripts.seed_data_generator import generate_all
    generate_all()

print(f"Processing video: {sample_clip}")
tracks_df = extract_tracks(sample_clip)
print(f"Extracted {len(tracks_df)} bounding box detections.")
tracks_df.head()

## 2. Engineer Motion & Velocity Features
Computes `avg_speed`, `max_speed`, `max_deceleration`, and `trajectory_variance` per vehicle track.

In [ ]:
motion_df = engineer_motion_features(tracks_df, fps=25)
print("Per-vehicle kinematic features:")
motion_df

## 3. Pairwise Bounding-Box Overlap (IOU)
Measures maximum intersection-over-union across detected vehicles in each frame.

In [ ]:
iou_df = iou_overlap_features(tracks_df)
print(f"Max overall IOU overlap: {iou_df['max_iou'].max():.3f}")
iou_df.head()

## 4. Aggregate Clip-Level Feature Row
Consolidates motion + IOU + environmental context into a single unscaled row matching `src.preprocessing` schema.

In [ ]:
context = {'road_type': 'highway', 'weather': 'clear', 'time_of_day': 'day'}
row_df = summarize_video_features(motion_df, iou_df, context)
print("Consolidated model input row:")
row_df